## Problem Formulation

**Domain:** Startup Analytics / Entrepreneurship  
**Problem Statement:**  Early identification of factors that influence startup success or failure.

**Main Question:**

- What factors most strongly predict whether a startup will succeed or fail?
- How profitable do successful startups tend to become, and what influences their level of profitability?
- How does the business domain (e.g., FinTech, HealthTech, AI, Retail) impact startup success rates and time to profitability?

## Plan Table
- Kaggle datasets
- one scrapped - google trends

| **Data Source**                                                                                                                            | **Type of Data**                                                                                         | **Collection Mode**    | **Format**        | **Sampling**                             | **Storage**         | **Description / Purpose**                                                                                     |
| ------------------------------------------------------------------------------------------------------------------------------------------ | -------------------------------------------------------------------------------------------------------- | ---------------------- | ----------------- | ---------------------------------------- | ------------------- | ------------------------------------------------------------------------------------------------------------- |
| [Startup Failure Prediction Dataset](https://www.kaggle.com/datasets/sakharebharat/startup-failure-prediction-dataset)                     | Quantitative (Funding_Amount, Revenue) + Qualitative (Market_Size, Startup_Status)                       | Downloaded             | `.csv`            | No                                       | Local storage / API | Used to analyze whether a startup can fail based on market and funding variables.                             |
| [Startup Growth and Funding Trends](https://www.kaggle.com/datasets/samayashar/startup-growth-and-funding-trends)                          | Quantitative (Funding_Amount, Growth_Rate, Valuation) + Qualitative (Industry, Location)                 | Downloaded             | `.csv`            | No                                       | Local storage / API | Used to analyze growth and profitability trends across industries and funding stages.                         |                        |
| Scraped / Custom Dataset                                         | Qualitative (Founder Experience, Risk Factors) + Quantitative (Years Active) | Scraped / Manual Entry | `.csv` | No                                       | Local storage       | Created to add updated attributes for a specific domain. |


In [ ]:
import pandas as pd

data = pd.read_csv("/content/startup_failure_prediction.csv")
data

In [ ]:
!pip install pytrends

In [ ]:
import time
import math
from datetime import datetime
from dateutil.relativedelta import relativedelta
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
from scipy import stats
from pytrends.request import TrendReq

# --------------------------
# CONFIG
# --------------------------
# Domains to analyze and their representative keyword sets.
DOMAINS: Dict[str, List[str]] = {
    "FinTech":   ["fintech startup", "financial technology startup"],
    "HealthTech":["healthtech startup", "digital health startup", "health technology startup"],
    "AI":        ["ai startup", "artificial intelligence startup"],
    "Retail":    ["retail startup", "ecommerce startup"],
}

# set to None for worldwide
GEO: Optional[str] = None

# Time window
START_DATE = "2016-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")  # today

# Resolution hint:
# - For long spans, Google returns weekly; we’ll convert to monthly for stability.
RESAMPLE_TO = "M"  # "M" for monthly; change to "W" for weekly if preferred

# Rate limiting / retries
SLEEP_SECONDS = 2.5
MAX_RETRIES = 3

# Output files
OUT_TIME = "trends_interest_over_time.csv"
OUT_REGION = "trends_interest_by_region.csv"
OUT_SUMMARY = "trends_summary_metrics.csv"

# --------------------------
# HELPERS
# --------------------------
def daterange_to_trends_timeframe(start: str, end: str) -> str:
    """Google Trends timeframe format: 'YYYY-MM-DD YYYY-MM-DD' or like 'today 5-y'."""
    return f"{start} {end}"

def safe_build_payload(pytrends: TrendReq, kw_list: List[str], timeframe: str, geo: Optional[str]) -> bool:
    """Builds payload with retries (to handle rate limiting)."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            pytrends.build_payload(kw_list=kw_list, timeframe=timeframe, geo=geo or "")
            return True
        except Exception as e:
            if attempt == MAX_RETRIES:
                print(f"[ERROR] build_payload failed after {MAX_RETRIES} attempts: {e}")
                return False
            sleep_for = SLEEP_SECONDS * attempt
            print(f"[WARN] build_payload error (attempt {attempt}): {e}. Sleeping {sleep_for:.1f}s...")
            time.sleep(sleep_for)
    return False

def safe_trends_call(callable_fn, label: str):
    """Wrap Trends calls with retries/backoff."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return callable_fn()
        except Exception as e:
            if attempt == MAX_RETRIES:
                print(f"[ERROR] {label} failed after {MAX_RETRIES} attempts: {e}")
                return None
            sleep_for = SLEEP_SECONDS * attempt
            print(f"[WARN] {label} error (attempt {attempt}): {e}. Sleeping {sleep_for:.1f}s...")
            time.sleep(sleep_for)
    return None

def monthly_resample(df: pd.DataFrame) -> pd.DataFrame:
    """Resample weekly/daily series to monthly mean; drop isPartial if present."""
    df2 = df.copy()
    if "isPartial" in df2.columns:
        df2 = df2.drop(columns=["isPartial"])
    # Ensure DateTimeIndex
    df2.index = pd.to_datetime(df2.index)
    # Monthly average
    return df2.resample(RESAMPLE_TO).mean().dropna(how="all")

def compute_metrics(series: pd.Series) -> Dict[str, float]:
    """
    Compute growth / momentum proxies from a normalized interest time series (0-100 scale).
    Returns:
      - cagr_pct
      - time_to_peak_months
      - recent_6m_slope
      - volatility
    """
    s = series.dropna()
    if s.empty:
        return dict(cagr_pct=np.nan, time_to_peak_months=np.nan, recent_6m_slope=np.nan, volatility=np.nan)

    # CAGR-like: interest_start->interest_end over full period (avoid zeros with +1)
    interest_start = s.iloc[0] + 1e-6
    interest_end = s.iloc[-1] + 1e-6
    n_years = max((s.index[-1].to_pydatetime() - s.index[0].to_pydatetime()).days / 365.25, 1e-6)
    cagr = (interest_end / interest_start) ** (1.0 / n_years) - 1.0

    # Time to peak interest (months since start)
    peak_idx = s.idxmax()
    time_to_peak_months = (peak_idx.to_pydatetime().year - s.index[0].year) * 12 + (peak_idx.month - s.index[0].month)

    # Recent 6-month momentum: slope of linear fit on last up-to-6 points
    tail = s.tail(6)
    if len(tail) >= 3:
        x = np.arange(len(tail))
        slope, _, _, _, _ = stats.linregress(x, tail.values)
        recent_6m_slope = float(slope)
    else:
        recent_6m_slope = np.nan

    # Volatility (std dev)
    volatility = float(np.std(s.values))

    return dict(
        cagr_pct=round(cagr * 100.0, 2),
        time_to_peak_months=float(time_to_peak_months),
        recent_6m_slope=round(recent_6m_slope, 4) if not math.isnan(recent_6m_slope) else np.nan,
        volatility=round(volatility, 2),
    )

# --------------------------
# MAIN
# --------------------------
def main():
    timeframe = daterange_to_trends_timeframe(START_DATE, END_DATE)
    pytrends = TrendReq(hl="en-US", tz=0, retries=0, backoff_factor=0)  # we manage retries ourselves

    all_time_tables = []
    all_region_tables = []
    metrics_rows = []

    for domain, keywords in DOMAINS.items():
        # Build payload for this domain
        print(f"\n=== Domain: {domain} | Keywords: {keywords} | GEO: {GEO or 'Worldwide'} ===")
        ok = safe_build_payload(pytrends, kw_list=keywords, timeframe=timeframe, geo=GEO)
        if not ok:
            print(f"[SKIP] Could not build payload for {domain}.")
            continue

        # Interest over time (multiple keywords). We ll combine them into a single domain series by taking the max per timestamp
        iot = safe_trends_call(pytrends.interest_over_time, "interest_over_time")
        if iot is None or iot.empty:
            print(f"[WARN] No interest-over-time data for {domain}.")
            continue

        # Resample to monthly and aggregate multiple keywords -> single domain series
        iot_m = monthly_resample(iot)
        # Combine columns (keywords) by max to represent domain interest envelope
        domain_series = iot_m[keywords].max(axis=1) if len(keywords) > 1 else iot_m[keywords[0]]
        domain_series.name = domain

        # Stash per-domain time series for a final wide table
        all_time_tables.append(domain_series)

        # Compute summary metrics
        metrics = compute_metrics(domain_series)
        metrics_rows.append({
            "domain": domain,
            **metrics
        })

        # Interest by region (top regions for this timeframe)
        ibr = safe_trends_call(lambda: pytrends.interest_by_region(resolution="country", inc_low_vol=True, inc_geo_code=True), "interest_by_region")
        if ibr is not None and not ibr.empty:
            ibr = ibr.reset_index().rename(columns={"geoName": "region"})
            # Aggregate across keywords using max (or mean)
            ibr["domain_interest"] = ibr[keywords].max(axis=1) if len(keywords) > 1 else ibr[keywords[0]]
            ibr = ibr[["region", "geoCode", "domain_interest"]]
            ibr["domain"] = domain
            all_region_tables.append(ibr)

        time.sleep(SLEEP_SECONDS)  # polite pause between domains

    # --------------------------
    # Export CSVs
    # --------------------------
    if all_time_tables:
        wide = pd.concat(all_time_tables, axis=1)
        wide.index.name = "date"
        wide.to_csv(OUT_TIME)
        print(f"[OK] Saved interest over time: {OUT_TIME}")
    else:
        print("[INFO] No time series to save.")

    if all_region_tables:
        regions_df = pd.concat(all_region_tables, axis=0, ignore_index=True)
        regions_df.to_csv(OUT_REGION, index=False)
        print(f"[OK] Saved interest by region: {OUT_REGION}")
    else:
        print("[INFO] No region data to save.")

    if metrics_rows:
        metrics_df = pd.DataFrame(metrics_rows).sort_values("cagr_pct", ascending=False)
        metrics_df.to_csv(OUT_SUMMARY, index=False)
        print(f"[OK] Saved summary metrics: {OUT_SUMMARY}")
        print(metrics_df)
    else:
        print("[INFO] No metrics computed.")

if __name__ == "__main__":
    main()



=== Domain: FinTech | Keywords: ['fintech startup', 'financial technology startup'] | GEO: Worldwide ===


/usr/local/lib/python3.12/dist-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)
/tmp/ipython-input-2375413043.py:88: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  return df2.resample(RESAMPLE_TO).mean().dropna(how="all")



=== Domain: HealthTech | Keywords: ['healthtech startup', 'digital health startup', 'health technology startup'] | GEO: Worldwide ===


/usr/local/lib/python3.12/dist-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)
/tmp/ipython-input-2375413043.py:88: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  return df2.resample(RESAMPLE_TO).mean().dropna(how="all")



=== Domain: AI | Keywords: ['ai startup', 'artificial intelligence startup'] | GEO: Worldwide ===


/usr/local/lib/python3.12/dist-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)
/tmp/ipython-input-2375413043.py:88: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  return df2.resample(RESAMPLE_TO).mean().dropna(how="all")



=== Domain: Retail | Keywords: ['retail startup', 'ecommerce startup'] | GEO: Worldwide ===


/usr/local/lib/python3.12/dist-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)
/tmp/ipython-input-2375413043.py:88: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  return df2.resample(RESAMPLE_TO).mean().dropna(how="all")


[OK] Saved interest over time: trends_interest_over_time.csv
[OK] Saved interest by region: trends_interest_by_region.csv
[OK] Saved summary metrics: trends_summary_metrics.csv
       domain  cagr_pct  time_to_peak_months  recent_6m_slope  volatility
2          AI    517.39                115.0          13.4286       13.05
1  HealthTech     37.62                115.0          13.2000       12.31
0     FinTech     23.08                115.0          12.3143       12.24
3      Retail     20.47                115.0          13.5143       12.02
